In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import joblib

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error
)

import lightgbm as lgb

import warnings
warnings.filterwarnings("ignore")

In [42]:
customerdf = pd.read_excel("output/Customer_Analytical.xlsx")

customer_productdf = pd.read_excel(
    "output/Customer_Product_Reorder.xlsx"
)

itemdf = pd.read_excel("output/Item_Analytical.xlsx")

coorderingdf = pd.read_excel(
    "output/Product_Coordering_Analysis.xlsx"
)

factdf = pd.read_csv(
    "output/Fact_Sales_Analytical.csv"
)

print("Customer:", customerdf.shape)
print("Customer × Product:", customer_productdf.shape)
print("Item:", itemdf.shape)
print("Co-ordering:", coorderingdf.shape)
print("Fact:", factdf.shape)

Customer: (2159, 26)
Customer × Product: (50959, 19)
Item: (5706, 31)
Co-ordering: (242025, 13)
Fact: (265135, 20)


In [154]:
customer = customerdf.copy()
customer_product = customer_productdf.copy()
item = itemdf.copy()
coordering = coorderingdf.copy()
fact = factdf.copy()

In [155]:
print("\nCUSTOMER")
print(customer.columns.tolist())

print("\nCUSTOMER PRODUCT")
print(customer_product.columns.tolist())

print("\nITEM")
print(item.columns.tolist())

print("\nCO-ORDERING")
print(coordering.columns.tolist())

print("\nFACT")
print(fact.columns.tolist())


CUSTOMER
['Customer Internal ID', 'Company Name', 'Customer Type', 'FA Sub Industries', 'total_revenue', 'total_quantity', 'total_orders', 'unique_products', 'first_order_date', 'last_order_date', 'average_order_value', 'recency_days', 'customer_lifetime_days', 'products_reordered', 'avg_reorder_interval_days', 'median_reorder_interval_days', 'reorderable_products', 'products_approaching_reorder', 'products_expected_now', 'products_overdue', 'recency_days_rfm', 'frequency', 'monetary', 'r_score', 'f_score', 'm_score']

CUSTOMER PRODUCT
['customer_internal_id', 'item_internal_id', 'item_code', 'first_order_date', 'last_order_date', 'order_count', 'total_quantity', 'avg_order_quantity', 'avg_reorder_interval_days', 'median_reorder_interval_days', 'reorder_interval_std', 'reorder_interval_count', 'days_since_last_order', 'reorder_interval_cv', 'reorder_predictability', 'reorder_confidence', 'expected_reorder_date', 'reorder_progress_pct', 'reorder_status']

ITEM
['Item Internal ID', 'Ite

In [156]:
fact["sales_order_date"] = pd.to_datetime(
    fact["sales_order_date"]
)

fact["year_month"] = pd.to_datetime(
    fact["year_month"]
)

fact = fact.sort_values(
    ["item_internal_id", "sales_order_date"]
).reset_index(drop=True)

print(
    "Date range:",
    fact["sales_order_date"].min(),
    "→",
    fact["sales_order_date"].max()
)

Date range: 2021-04-01 00:00:00 → 2025-03-31 00:00:00


In [157]:
fact["week"] = (
    fact["sales_order_date"]
    .dt.to_period("W")
    .dt.start_time
)

weekly_demand = (
    fact.groupby(
        ["item_internal_id", "week"],
        as_index=False
    )
    .agg(
        demand=("quantity", "sum"),
        revenue=("net_revenue", "sum"),
        orders=("order_key", "nunique"),
        customers=("customer_internal_id", "nunique")
    )
)

weekly_demand.head()

,item_internal_id,week,demand,revenue,orders,customers
0,111,2023-11-20,1,4000.75,1,1
1,111,2023-12-25,2,8001.50,1,1
2,2659,2022-05-30,1,40326.00,1,1
3,2659,2022-10-03,1,40326.00,1,1
4,6199,2021-03-29,90,122710.00,2,2


In [158]:
demand = 0
all_weeks = pd.date_range(
    start="2020-01-06",
    end="2025-3-31",
    freq="W-MON"
)

all_products = (
    fact["item_internal_id"]
    .dropna()
    .unique()
)

complete_index = pd.MultiIndex.from_product(
    [all_products, all_weeks],
    names=["item_internal_id", "week"]
)

weekly_demand = (
    weekly_demand
    .set_index(
        ["item_internal_id", "week"]
    )
    .reindex(complete_index)
    .reset_index()
)

weekly_demand["demand"] = (
    weekly_demand["demand"]
    .fillna(0)
)

weekly_demand["revenue"] = (
    weekly_demand["revenue"]
    .fillna(0)
)

weekly_demand["orders"] = (
    weekly_demand["orders"]
    .fillna(0)
)

weekly_demand["customers"] = (
    weekly_demand["customers"]
    .fillna(0)
)

weekly_demand.head()

,item_internal_id,week,demand,revenue,orders,customers
0,111,2020-01-06,0.0,0.0,0.0,0.0
1,111,2020-01-13,0.0,0.0,0.0,0.0
2,111,2020-01-20,0.0,0.0,0.0,0.0
3,111,2020-01-27,0.0,0.0,0.0,0.0
4,111,2020-02-03,0.0,0.0,0.0,0.0


In [159]:
weekly_demand = weekly_demand.sort_values(
    ["item_internal_id", "week"]
)

grouped = weekly_demand.groupby(
    "item_internal_id"
)

# Lag features
for lag in [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 24, 36, 48, 52]:
    weekly_demand[f"demand_lag_{lag}"] = (
        grouped["demand"]
        .shift(lag)
    )

# Rolling demand
for window in [2, 4, 6, 8, 12, 26, 52]:

    weekly_demand[f"demand_mean_{window}w"] = (
        grouped["demand"]
        .shift(1)
        .rolling(window)
        .mean()
        .reset_index(level=0, drop=True)
    )

    weekly_demand[f"demand_std_{window}w"] = (
        grouped["demand"]
        .shift(1)
        .rolling(window)
        .std()
        .reset_index(level=0, drop=True)
    )

In [160]:
weekly_demand["demand_trend_2w"] = (
    weekly_demand["demand_lag_1"] -
    weekly_demand["demand_lag_2"]
)

weekly_demand["demand_trend_4w"] = (
    weekly_demand["demand_lag_1"] -
    weekly_demand["demand_lag_4"]
)

weekly_demand["demand_trend_8w"] = (
    weekly_demand["demand_lag_1"] -
    weekly_demand["demand_lag_8"]
)

weekly_demand["demand_trend_12w"] = (
    weekly_demand["demand_lag_1"] -
    weekly_demand["demand_lag_12"]
)

In [161]:
weekly_demand = weekly_demand.sort_values(
    ["item_internal_id", "week"]
).reset_index(drop=True)

weekly_demand["is_zero_demand"] = (
    weekly_demand["demand"] == 0
).astype(int)

for window in [2, 4, 6, 12]:

    weekly_demand[f"zero_demand_{window}w"] = (
        weekly_demand
        .groupby("item_internal_id")["is_zero_demand"]
        .transform(
            lambda x: x.shift(1).rolling(window).sum()
        )
    )

print("Done.")

Done.


In [162]:
weekly_demand["last_nonzero_demand"] = (
    weekly_demand
    .groupby("item_internal_id")["demand"]
    .transform(
        lambda x: x.shift(1).replace(0, np.nan).ffill()
    )
)

In [163]:
def weeks_since_nonzero(series):
    result = []
    counter = np.nan

    for value in series:
        if value > 0:
            counter = 0
        elif pd.notna(counter):
            counter += 1

        result.append(counter)

    return pd.Series(
        result,
        index=series.index
    )

weekly_demand["weeks_since_demand"] = (
    weekly_demand
    .groupby("item_internal_id")["demand"]
    .transform(weeks_since_nonzero)
)

In [164]:
weekly_demand["demand_same_week_last_year"] = (
    weekly_demand
    .groupby("item_internal_id")["demand"]
    .shift(52)
)

In [165]:
weekly_demand["yoy_demand_change"] = (
    weekly_demand["demand_lag_1"] -
    weekly_demand["demand_same_week_last_year"]
)

In [166]:
weekly_demand["month"] = (
    weekly_demand["week"].dt.month
)

weekly_demand["quarter"] = (
    weekly_demand["week"].dt.quarter
)

weekly_demand["week_of_year"] = (
    weekly_demand["week"].dt.isocalendar().week.astype(int)
)

weekly_demand["year"] = (
    weekly_demand["week"].dt.year
)

In [168]:
# Keep reorder predictability as a categorical variable
customer_product["reorder_predictability"] = (
    customer_product["reorder_predictability"].astype("category")
)

# Aggregate predictability by product
cp_predictability = (
    customer_product
    .groupby("item_internal_id", observed=True)
    .agg(
        highly_predictable_customers=(
            "reorder_predictability",
            lambda x: (x == "Highly Predictable").sum()
        ),
        moderately_predictable_customers=(
            "reorder_predictability",
            lambda x: (x == "Moderately Predictable").sum()
        ),
        unpredictable_customers=(
            "reorder_predictability",
            lambda x: (x == "Unpredictable").sum()
        ),
        insufficient_history_customers=(
            "reorder_predictability",
            lambda x: (x == "Insufficient History").sum()
        ),
        total_reorder_interval_records=(
            "reorder_interval_count",
            "sum"
        )
    )
    .reset_index()
)
weekly_demand = weekly_demand.merge(
    cp_predictability,
    on="item_internal_id",
    how="left"
)

In [169]:
item.columns

Index(['Item Internal ID', 'Item Code', 'Production Status',
       'Product Category 1', 'Product Category 2', 'List price',
       'total_quantity', 'total_revenue', 'total_orders', 'unique_customers',
       'first_sale_date', 'last_sale_date', 'avg_units_per_day',
       'avg_units_per_month', 'monthly_demand_mean', 'monthly_demand_std',
       'demand_cv', 'xyz_class', 'revenue_share', 'cumulative_revenue_share',
       'abc_class', 'reorder_customers', 'avg_reorder_interval_days',
       'median_reorder_interval_days', 'reorder_interval_std',
       'customers_approaching_reorder', 'customers_expected_now',
       'customers_overdue', 'is_discontinued', 'is_active_product',
       'discontinued_demand_risk'],
      dtype='str')

In [170]:
# Copy item analytical data
item_features = item.copy()

# Rename columns to match the forecasting dataset
item_features = item_features.rename(
    columns={
        "Item Internal ID": "item_internal_id",
        "Item Code": "item_code",
        "Production Status": "production_status",
        "Product Category 1": "product_category_1",
        "Product Category 2": "product_category_2"
    }
)

# Keep the product-level features we want
item_features = item_features[
    [
        "item_internal_id",
        "production_status",
        "product_category_1",
        "product_category_2",
        "total_orders",
        "unique_customers",
        "monthly_demand_mean",
        "monthly_demand_std",
        "demand_cv",
        "xyz_class",
        "abc_class",
        "reorder_customers",
        "avg_reorder_interval_days",
        "median_reorder_interval_days",
        "reorder_interval_std",
        "is_discontinued",
        "is_active_product",
        "discontinued_demand_risk"
    ]
].copy()

print("item_features shape:", item_features.shape)
item_features.head()

item_features shape: (5706, 18)


,item_internal_id,production_status,product_category_1,product_category_2,total_orders,unique_customers,monthly_demand_mean,monthly_demand_std,demand_cv,xyz_class,abc_class,reorder_customers,avg_reorder_interval_days,median_reorder_interval_days,reorder_interval_std,is_discontinued,is_active_product,discontinued_demand_risk
0,25214,Continue,Robot,Robot Others,4,3,1.333333,0.577350,0.433013,X,C,3,0.0,0.0,NaN,0,1,Normal
1,113835,Continue,Robot,Robot Others,4,3,2.500000,0.707107,0.282843,X,C,3,0.0,0.0,NaN,0,1,Normal
2,20722,Continue,iQSS,Anywire,2,1,6.000000,NaN,NaN,Insufficient Data,C,1,0.0,0.0,NaN,0,1,Normal
3,9754,Discontinue,PLC,PLC CC-Link,4,3,2.666667,1.527525,0.572822,Y,C,3,0.0,0.0,NaN,1,0,Review
4,9790,Continue,PLC,PLC CC-Link,6,5,3.250000,3.201562,0.985096,Y,C,5,0.0,0.0,NaN,0,1,Normal


In [171]:
weekly_demand = weekly_demand.merge(
    item_features,
    on="item_internal_id",
    how="left"
)

In [172]:
cp = customer_product.copy()

cp["last_order_date"] = pd.to_datetime(
    cp["last_order_date"]
)

cp["expected_reorder_date"] = pd.to_datetime(
    cp["expected_reorder_date"]
)

In [173]:
def calculate_reorder_features(
    historical_date,
    cp_data
):

    temp = cp_data[
        cp_data["last_order_date"] <= historical_date
    ].copy()

    temp["days_since_last_order"] = (
        historical_date -
        temp["last_order_date"]
    ).dt.days

    temp["days_until_reorder"] = (
        temp["expected_reorder_date"] -
        historical_date
    ).dt.days

    # Customer expected to reorder within next 30 days
    temp["approaching"] = (
        (temp["days_until_reorder"] >= 0) &
        (temp["days_until_reorder"] <= 30)
    ).astype(int)

    # Already overdue
    temp["overdue"] = (
        temp["days_until_reorder"] < 0
    ).astype(int)

    result = (
        temp.groupby("item_internal_id")
        .agg(
            customers_approaching_reorder=(
                "approaching",
                "sum"
            ),
            customers_overdue=(
                "overdue",
                "sum"
            ),
            avg_days_since_customer_order=(
                "days_since_last_order",
                "mean"
            ),
            median_days_since_customer_order=(
                "days_since_last_order",
                "median"
            )
        )
        .reset_index()
    )

    return result

In [174]:
weekly_reorder_features = []

for date in weekly_demand["week"].unique():

    features = calculate_reorder_features(
        date,
        cp
    )

    features["week"] = date

    weekly_reorder_features.append(
        features
    )

weekly_reorder_features = pd.concat(
    weekly_reorder_features,
    ignore_index=True
)

In [175]:
weekly_demand = weekly_demand.merge(
    weekly_reorder_features,
    on=["item_internal_id", "week"],
    how="left"
)

In [176]:
reorder_columns = [
    "customers_approaching_reorder",
    "customers_overdue",
    "avg_days_since_customer_order",
    "median_days_since_customer_order"
]

weekly_demand[reorder_columns] = (
    weekly_demand[reorder_columns]
    .fillna(0)
)

In [178]:
weekly_demand = weekly_demand[
    weekly_demand["demand_lag_52"].notna()
].copy()


In [189]:
train = weekly_demand[
    weekly_demand["week"] < "2024-12-31"
].copy()

test = weekly_demand[
    (weekly_demand["week"] >= "2024-01-01") & (weekly_demand["week"] <= "2025-3-31")
].copy()

print("Train:", train.shape)
print("Test:", test.shape)

print(
    train["week"].min(),
    "→",
    train["week"].max()
)

print(
    test["week"].min(),
    "→",
    test["week"].max()
)

Train: (1192554, 79)
Test: (376596, 79)
2021-01-04 00:00:00 → 2024-12-30 00:00:00
2024-01-01 00:00:00 → 2025-03-31 00:00:00


In [190]:
features = [
    # Historical demand
    "demand_lag_1",
    "demand_lag_2",
    "demand_lag_3",
    "demand_lag_4",
    "demand_lag_5",
    "demand_lag_6",
    "demand_lag_7",
    "demand_lag_8",
    "demand_lag_9",
    "demand_lag_10",
    "demand_lag_11",
    "demand_lag_12",
    "demand_lag_24",
    "demand_lag_36",
    "demand_lag_48",
    "demand_lag_52",

    # Rolling demand
    "demand_mean_2w",
    "demand_mean_4w",
    "demand_mean_6w",   
    "demand_mean_8w",
    "demand_mean_12w",
    "demand_mean_26w",
    "demand_mean_52w",

    "demand_std_2w",
    "demand_std_4w",
    "demand_std_6w",   
    "demand_std_8w",
    "demand_std_12w",
    "demand_std_26w",
    "demand_std_52w",

    # Trends
    "demand_trend_2w",
    "demand_trend_4w",
    "demand_trend_8w",
    "demand_trend_12w",

    # Zero demand features
    "zero_demand_2w",
    "zero_demand_4w",
    "zero_demand_6w",
    "zero_demand_12w",

    # Last non-zero demand
    "last_nonzero_demand",

    # Weeks since last non-zero demand
    "weeks_since_demand",

    # Year-over-year demand change
    "yoy_demand_change",

    # Current activity
    "orders",
    "customers",

    # Product
    "production_status",
    "product_category_1",
    "product_category_2",
    "total_orders",
    "unique_customers",
    "monthly_demand_mean",
    "monthly_demand_std",
    "demand_cv",
    "xyz_class",
    "abc_class",
    "reorder_customers",
    "avg_reorder_interval_days",
    "median_reorder_interval_days",
    "reorder_interval_std",

    # Reorder behavior
    "customers_approaching_reorder",
    "customers_overdue",
    "avg_days_since_customer_order",
    "median_days_since_customer_order",
    "highly_predictable_customers",
    "moderately_predictable_customers",
    "unpredictable_customers",
    "insufficient_history_customers",
    "total_reorder_interval_records",

    # Calendar
    "month",
    "quarter",
    "week_of_year"
]

In [191]:
categorical_features = [
    col
    for col in [
        "production_status",
        "product_category_1",
        "product_category_2",
        "xyz_class",
        "abc_class"
    ]
    if col in train.columns and col in features
]

for col in categorical_features:
    train[col] = train[col].astype("category")
    test[col] = test[col].astype("category")

In [192]:
# Use the actual weekly demand column as the prediction target.
# 'target_demand' was never created in the feature-building pipeline.
# This model predicts the observed weekly demand for each product/week.
X_train = train[features]
y_train = train["demand"]

X_test = test[features]
y_test = test["demand"]

In [193]:
model = lgb.LGBMRegressor(
    objective="regression",

    # More boosting rounds with a smaller learning step
    n_estimators=5000,
    learning_rate=0.01,

    # More complex trees
    num_leaves=63,
    max_depth=-1,

    # Minimum observations per leaf
    min_child_samples=30,

    # Randomization / regularization
    subsample=0.85,
    subsample_freq=1,
    colsample_bytree=0.85,

    # L1 / L2 regularization
    reg_alpha=0.1,
    reg_lambda=0.5,

    # Handle potentially noisy demand
    max_bin=255,

    random_state=42,
    n_jobs=-1
)

In [194]:
model.fit(
    X_train,
    y_train,
    categorical_feature=categorical_features,
    eval_set=[(X_test, y_test)],
    callbacks=[
        lgb.early_stopping(100),
        lgb.log_evaluation(100)
    ]
)

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.101778 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 9865
[LightGBM] [Info] Number of data points in the train set: 1192554, number of used features: 69
[LightGBM] [Info] Start training from score 0.693955
Training until validation scores don't improve for 100 rounds
[100]	valid_0's l2: 136.666
[200]	valid_0's l2: 119.33
[300]	valid_0's l2: 106.334
[400]	valid_0's l2: 96.6181
[500]	valid_0's l2: 90.808
[600]	valid_0's l2: 86.4307
[700]	valid_0's l2: 82.3536
[800]	valid_0's l2: 78.4829
[900]	valid_0's l2: 74.8605
[1000]	valid_0's l2: 71.2511
[1100]	valid_0's l2: 68.6714
[1200]	valid_0's l2: 65.5409
[1300]	valid_0's l2: 62.8949
[1400]	valid_0's l2: 60.3389
[1500]	valid_0's l2: 57.6416
[1600]	valid_0's l2: 55.1636
[1700]	valid_0's l2: 53.3717
[1800]	valid_0's l2: 51.501
[1900]	valid_0's l2:

,num_leaves,63
,learning_rate,0.01
,n_estimators,5000
,objective,'regression'
,min_child_samples,30
,subsample,0.85
,subsample_freq,1
,colsample_bytree,0.85
,reg_alpha,0.1
,reg_lambda,0.5
,random_state,42


In [195]:
predictions = model.predict(
    X_test
)

predictions = np.maximum(
    predictions,
    0
)

In [196]:
mae = mean_absolute_error(
    y_test,
    predictions
)

rmse = np.sqrt(
    mean_squared_error(
        y_test,
        predictions
    )
)

print(f"MAE:  {mae:.2f}")
print(f"RMSE: {rmse:.2f}")

MAE:  0.32
RMSE: 4.56


In [ ]:
results = test[
    [
        "week",
        "item_internal_id",
        "demand"
    ]
].copy()

results = results.rename(columns={"demand": "actual_demand"})
results["predicted_demand"] = predictions

results["absolute_error"] = (
    results["actual_demand"] -
    results["predicted_demand"]
).abs()

results["squared_error"] = (
    results["actual_demand"] -
    results["predicted_demand"]
) ** 2

results.to_excel(
    "LightGBM_Weekly_Demand_Predictions.xlsx",
    index=False
)

In [201]:
with open("output/mitsubishi_lightgbm_categorical_features.json", "w") as f:
    json.dump(categorical_features, f)

In [203]:
final_train = weekly_demand[
    weekly_demand["week"] < "2025-03-31"
].copy()

X_final = final_train[features]
y_final = final_train["demand"]

In [204]:
for col in categorical_features:
    X_final[col] = X_final[col].astype("category")

In [205]:
final_model = lgb.LGBMRegressor(
    objective="regression",

    n_estimators=6000,
    learning_rate=0.01,

    num_leaves=63,
    max_depth=-1,
    min_child_samples=30,

    subsample=0.85,
    subsample_freq=1,
    colsample_bytree=0.85,

    reg_alpha=0.1,
    reg_lambda=0.5,

    max_bin=255,

    random_state=42,
    n_jobs=-1
)

In [208]:
final_model.fit(
    X_final,
    y_final,
    categorical_feature=categorical_features,
    callbacks=[
        lgb.log_evaluation(100)
    ]
)

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.113892 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 9836
[LightGBM] [Info] Number of data points in the train set: 1261026, number of used features: 69
[LightGBM] [Info] Start training from score 0.689081


,num_leaves,63
,learning_rate,0.01
,n_estimators,6000
,objective,'regression'
,min_child_samples,30
,subsample,0.85
,subsample_freq,1
,colsample_bytree,0.85
,reg_alpha,0.1
,reg_lambda,0.5
,random_state,42


In [209]:
joblib.dump(
    model,
    "output/mitsubishi_lightgbm_demand_model.pkl"
)

print("Model exported successfully.")

Model exported successfully.


In [ ]:
weekly_demand.to_csv(
    "output/weekly_demand_features.csv",
    index=False
)

In [212]:
item_features.to_csv(
    "output/item_features.csv",
    index=False
)

In [200]:
import json

with open("output/mitsubishi_lightgbm_features.json", "w") as f:
    json.dump(features, f)

print("Feature list exported.")

Feature list exported.
